In [1]:
!pip install -q protobuf==3.20.3

!pip install -q torchnet tensorboardX einops gdown wandb

import time
import os
import sys
import json
import math
import random
import re
import csv
import shutil
import tarfile
import tempfile
import datetime
import warnings
import numpy as np
import cv2 as cv
import gdown
from PIL import Image
from tqdm.auto import tqdm
from os import listdir
from os.path import isfile, isdir, join
from abc import abstractmethod
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
from torchvision.utils import make_grid


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, bu

2025-11-21 04:10:57.270309: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763698257.478192      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763698257.541671      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


Using device: cuda:0


In [3]:
!mkdir -p miniimagenet

%cd miniimagenet


/kaggle/working/miniimagenet


In [4]:
!wget https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet/train.csv -O train.csv

!wget https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet/val.csv -O val.csv

!wget https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet/test.csv -O test.csv


--2025-11-21 04:11:13--  https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet/train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1228815 (1.2M) [text/plain]
Saving to: ‘train.csv’

train.csv           100%[===================>]   1.17M  --.-KB/s    in 0.03s   

2025-11-21 04:11:13 (39.5 MB/s) - ‘train.csv’ saved [1228815/1228815]

--2025-11-21 04:11:13--  https://raw.githubusercontent.com/twitter/meta-learning-lstm/master/data/miniImagenet/val.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Leng

In [5]:
!pip install -q gdown

DATASETS_DIR = '/kaggle/working/miniimagenet/Datasets'
os.makedirs(DATASETS_DIR, exist_ok=True)
existing_classes = [d for d in os.listdir(DATASETS_DIR) if d.startswith('n') and os.path.isdir(os.path.join(DATASETS_DIR, d))]
if len(existing_classes) > 0:
    print('[✓] Dataset already exists.')
    print(f'[✓] Found {len(existing_classes)} class folders in: {DATASETS_DIR}')
    print('[✓] Skipping download and extraction.')
else:
    print('[*] Dataset not found — starting download + extract workflow.')
    temp_dir = tempfile.TemporaryDirectory()
    temp_download = os.path.join(temp_dir.name, 'download')
    temp_extract = os.path.join(temp_dir.name, 'extract')
    os.makedirs(temp_download, exist_ok=True)
    os.makedirs(temp_extract, exist_ok=True)
    print('Temporary folder created at:', temp_dir.name)
    FOLDER_URL = 'https://drive.google.com/drive/folders/17a09kkqVivZQFggCw9I_YboJ23tcexNM'
    print('[*] Downloading tar files into the temporary directory...')
    gdown.download_folder(url=FOLDER_URL, output=temp_download, quiet=False, use_cookies=False)
    print('[+] Download complete.')
    print('[*] Extracting tar archives...')
    for fname in os.listdir(temp_download):
        if fname.endswith('.tar'):
            tar_path = os.path.join(temp_download, fname)
            print(f'  - Extracting: {tar_path}')
            with tarfile.open(tar_path, 'r') as tar:
                tar.extractall(temp_extract)
    print('[+] Extraction complete.')
    print('[*] Moving class folders to Datasets/...')
    for (root, dirs, files) in os.walk(temp_extract):
        for d in dirs:
            if d.startswith('n'):
                src = os.path.join(root, d)
                dst = os.path.join(DATASETS_DIR, d)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
    print('[+] All done!')
    print('Final dataset path:', DATASETS_DIR)


[*] Dataset not found — starting download + extract workflow.
Temporary folder created at: /tmp/tmpsvm1_bmw
[*] Downloading tar files into the temporary directory...


Retrieving folder contents


Processing file 1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v test.tar
Processing file 107FTosYIeBn5QbynR46YG91nHcJ70whs train.tar
Processing file 1hSMUMj5IRpf-nQs1OwgiQLmGZCN0KDWl val.tar


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v
From (redirected): https://drive.google.com/uc?id=1yKyKgxcnGMIAnA_6Vr2ilbpHMc9COg-v&confirm=t&uuid=5028da98-4679-4550-9125-dd0714b8cf59
To: /tmp/tmpsvm1_bmw/download/test.tar
100%|██████████| 39.2M/39.2M [00:00<00:00, 75.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=107FTosYIeBn5QbynR46YG91nHcJ70whs
From (redirected): https://drive.google.com/uc?id=107FTosYIeBn5QbynR46YG91nHcJ70whs&confirm=t&uuid=bd6b6ae6-560b-479d-9ef1-c82cc0f4d73d
To: /tmp/tmpsvm1_bmw/download/train.tar
100%|██████████| 126M/126M [00:01<00:00, 111MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1hSMUMj5IRpf-nQs1OwgiQLmGZCN0KDWl
From (redirected): https://drive.google.com/uc?id=1hSMUMj5IRpf-nQs1OwgiQLmGZCN0KDWl&confirm=t&uuid=007e8328-5da8-42e7-9b60-f7f516f90356
To: /tmp/tmpsv

[+] Download complete.
[*] Extracting tar archives...
  - Extracting: /tmp/tmpsvm1_bmw/download/train.tar
  - Extracting: /tmp/tmpsvm1_bmw/download/val.tar
  - Extracting: /tmp/tmpsvm1_bmw/download/test.tar
[+] Extraction complete.
[*] Moving class folders to Datasets/...
[+] All done!
Final dataset path: /kaggle/working/miniimagenet/Datasets


In [6]:
%cd /kaggle/working/miniimagenet

cwd = os.getcwd()
print(f'Current Working Directory (cwd): {cwd}')
data_path = join(cwd, 'Datasets')
savedir = './'
dataset_list = ['base', 'val', 'novel']
cl = -1
folderlist = []
datasetmap = {'base': 'train', 'val': 'val', 'novel': 'test'}
filelists = {'base': {}, 'val': {}, 'novel': {}}
filelists_flat = {'base': [], 'val': [], 'novel': []}
labellists_flat = {'base': [], 'val': [], 'novel': []}
for dataset in dataset_list:
    with open(datasetmap[dataset] + '.csv', 'r') as lines:
        for (i, line) in enumerate(lines):
            if i == 0:
                continue
            (fid, _, label) = re.split(',|\\.', line)
            label = label.replace('\n', '')
            if not label in filelists[dataset]:
                folderlist.append(label)
                filelists[dataset][label] = []
                fnames = listdir(join(data_path, label))
            name = fid + '.jpg'
            fname = join(data_path, label, name)
            filelists[dataset][label].append(fname)
    for (key, filelist) in filelists[dataset].items():
        cl += 1
        random.shuffle(filelist)
        filelists_flat[dataset] += filelist
        labellists_flat[dataset] += np.repeat(cl, len(filelist)).tolist()
for dataset in dataset_list:
    fo = open(savedir + dataset + '.json', 'w')
    fo.write('{"label_names": [')
    fo.writelines(['"%s",' % item for item in folderlist])
    fo.seek(0, os.SEEK_END)
    fo.seek(fo.tell() - 1, os.SEEK_SET)
    fo.write('],')
    fo.write('"image_names": [')
    fo.writelines(['"%s",' % item for item in filelists_flat[dataset]])
    fo.seek(0, os.SEEK_END)
    fo.seek(fo.tell() - 1, os.SEEK_SET)
    fo.write('],')
    fo.write('"image_labels": [')
    fo.writelines(['%d,' % item for item in labellists_flat[dataset]])
    fo.seek(0, os.SEEK_END)
    fo.seek(fo.tell() - 1, os.SEEK_SET)
    fo.write(']}')
    fo.close()
    print('%s -OK' % dataset)


/kaggle/working/miniimagenet
Current Working Directory (cwd): /kaggle/working/miniimagenet
base -OK
val -OK
novel -OK


In [7]:
class configs:
    save_dir = ''
    data_dir = '/kaggle/working/miniimagenet/'


In [8]:
from PIL import Image

class MiniImagenet(Dataset):

    def __init__(self, root, mode, n_way, k_shot, k_query, batchsz, resize, startidx=0):
        self.batchsz = batchsz
        self.n_way = n_way
        self.k_shot = k_shot
        self.k_query = k_query
        self.resize = resize
        json_file_name = f'{mode}.json'
        json_path = os.path.join(root, json_file_name)
        if mode == 'base' and (not os.path.exists(json_path)):
            json_path = os.path.join(root, 'train.json')
        if not os.path.exists(json_path):
            raise FileNotFoundError(f'JSON file not found at: {json_path}. Vui lòng kiểm tra tên file trong thư mục Datasets.')
        print(f'Loading data from {json_path}...')
        with open(json_path, 'r') as f:
            data = json.load(f)
        self.path = data['image_names']
        self.label = data['image_labels']
        self.img2label = {}
        for (path, label) in zip(self.path, self.label):
            if label not in self.img2label:
                self.img2label[label] = []
            self.img2label[label].append(path)
        self.cls_num = len(self.img2label)
        self.label_keys = sorted(list(self.img2label.keys()))
        self.transform = transforms.Compose([transforms.Resize((resize, resize)), transforms.ToTensor(), transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

    def __getitem__(self, index):
        selected_cls_keys = np.random.choice(self.label_keys, self.n_way, replace=False)
        support_x = []
        support_y = []
        query_x = []
        query_y = []
        for (i, cls_key) in enumerate(selected_cls_keys):
            imgs_path = self.img2label[cls_key]
            num_samples_needed = self.k_shot + self.k_query
            replace_sampling = len(imgs_path) < num_samples_needed
            selected_imgs_idx = np.random.choice(len(imgs_path), num_samples_needed, replace=replace_sampling)
            for (j, idx) in enumerate(selected_imgs_idx):
                path = imgs_path[idx]
                img = Image.open(path).convert('RGB')
                img = self.transform(img)
                if j < self.k_shot:
                    support_x.append(img)
                    support_y.append(i)
                else:
                    query_x.append(img)
                    query_y.append(i)
        support_x = torch.stack(support_x)
        query_x = torch.stack(query_x)
        support_y = torch.tensor(support_y).long()
        query_y = torch.tensor(query_y).long()
        return (support_x, support_y, query_x, query_y)

    def __len__(self):
        return self.batchsz

def make_imgs_visual(n_way, k_shot, k_query, support_x, support_y, query_x, query_y, pred, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    device = support_x.device
    mean = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std = torch.tensor(std, device=device).view(1, 3, 1, 1)
    sup = support_x[0] * std + mean
    qry = query_x[0] * std + mean
    s_y = support_y[0]
    idx = torch.argsort(s_y)
    sup = sup[idx]
    grid_sup = make_grid(sup, nrow=k_shot)
    grid_qry = make_grid(qry, nrow=k_query)
    return (grid_sup, grid_qry)


# --- Model Architecture (ResNet + Relation Net) ---


In [9]:
class ConvBlock(nn.Module):
    """Convolutional Block theo Figure 2a của bài báo."""

    def __init__(self, in_channels, use_maxpool=True):
        super(ConvBlock, self).__init__()
        self.layer = nn.Sequential(nn.Conv2d(in_channels, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.maxpool = nn.MaxPool2d(kernel_size=2) if use_maxpool else None

    def forward(self, x):
        x = self.layer(x)
        if self.maxpool:
            x = self.maxpool(x)
        return x

class EmbeddingModule(nn.Module):
    """
    4 Convolutional Blocks (Figure 2b). Hai khối đầu có MaxPool.
    """

    def __init__(self):
        super(EmbeddingModule, self).__init__()
        self.blocks = nn.Sequential(ConvBlock(3, use_maxpool=True), ConvBlock(64, use_maxpool=True), ConvBlock(64, use_maxpool=False), ConvBlock(64, use_maxpool=False))

    def forward(self, x):
        return self.blocks(x)

class RelationModule(nn.Module):
    """
    Relation Module g_phi theo Section 3.4.
    """

    def __init__(self, input_channels):
        super(RelationModule, self).__init__()
        self.conv_block1 = nn.Sequential(nn.Conv2d(input_channels, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=2))
        self.conv_block2 = nn.Sequential(nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=2))
        final_flat_size = 1600
        self.fc_layers = nn.Sequential(nn.Linear(final_flat_size, 8), nn.ReLU(inplace=True), nn.Linear(8, 1), nn.Sigmoid())

    def forward(self, x):
        out = self.conv_block1(x)
        out = self.conv_block2(out)
        out = out.view(out.size(0), -1)
        score = self.fc_layers(out)
        return score


In [11]:
class Compare(nn.Module):
    """Relation Network (RN) - Học ít dữ liệu"""

    def __init__(self, n_way, k_shot, resize=None):
        super(Compare, self).__init__()
        current_resize = RESIZE if resize is None else resize
        self.resize = current_resize
        self.n_way = n_way
        self.k_shot = k_shot
        self.f_phi = EmbeddingModule()
        with torch.no_grad():
            dummy = torch.randn(1, 3, current_resize, current_resize)
            feat = self.f_phi(dummy)
        self.c = feat.size(1)
        self.d = feat.size(2)
        self.relation_input_channels = 2 * self.c
        self.g_phi = RelationModule(self.relation_input_channels)
        print(f'RN Feature Map Size: {self.c}x{self.d}x{self.d}')

    def forward(self, support_x, support_y, query_x, query_y, train=True):
        (batchsz, setsz, c_, h, w) = support_x.size()
        querysz = query_x.size(1)
        support_xf = self.f_phi(support_x.view(-1, c_, h, w))
        query_xf = self.f_phi(query_x.view(-1, c_, h, w)).view(batchsz, querysz, self.c, self.d, self.d)
        class_features = []
        for b in range(batchsz):
            unique_labels = torch.unique(support_y[b])
            for cls_idx in unique_labels:
                indices = (support_y[b] == cls_idx).nonzero(as_tuple=True)[0]
                feats = support_xf[b * setsz + indices]
                class_feats_sum = torch.sum(feats, dim=0).unsqueeze(0)
                class_features.append(class_feats_sum)
        class_xf = torch.cat(class_features, dim=0)
        class_xf = class_xf.view(batchsz, self.n_way, self.c, self.d, self.d)
        class_xf_expanded = class_xf.unsqueeze(1).expand(-1, querysz, -1, -1, -1, -1)
        query_xf_expanded = query_xf.unsqueeze(2).expand(-1, -1, self.n_way, -1, -1, -1)
        comb = torch.cat([query_xf_expanded, class_xf_expanded], dim=3)
        score = self.g_phi(comb.view(-1, self.relation_input_channels, self.d, self.d)).view(batchsz, querysz, self.n_way)
        query_y_expanded = query_y.unsqueeze(2).expand(batchsz, querysz, self.n_way)
        class_labels_for_comparison = torch.arange(self.n_way, device=DEVICE).long().view(1, 1, self.n_way)
        label = torch.eq(query_y_expanded, class_labels_for_comparison).float()
        if train:
            loss = torch.pow(label - score, 2).sum() / batchsz
            return loss
        else:
            (_, pred_idx) = torch.max(score, dim=2)
            correct = torch.eq(pred_idx, query_y).sum()
            return (pred_idx, correct)


In [15]:
N_WAY = 5
K_SHOT = 1
K_QUERY = 15
BATCH_SIZE = 1
EPOCHS = 50
RESIZE = 84


In [16]:
ROOT_DIR = os.getcwd()
train_set = MiniImagenet(ROOT_DIR, 'base', N_WAY, K_SHOT, K_QUERY, 3000, RESIZE)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
val_set = MiniImagenet(ROOT_DIR, 'val', N_WAY, K_SHOT, K_QUERY, 600, RESIZE)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
model = Compare(N_WAY, K_SHOT, resize=RESIZE).to(DEVICE)
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
mdl_file = f'relation_net_{N_WAY}w{K_SHOT}s.pth'
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
tb = SummaryWriter('runs/final_relation_net')
print('Start Training...')
best_acc = 0.0
global_best_test_acc = 0.0
global_best_test_loss = float('inf')
for epoch in range(EPOCHS):
    start_time = time.time()
    model.train()
    total_train_loss = 0
    total_train_correct = 0
    total_train_items = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS} (Train)')
    for (step, (sup_x, sup_y, qry_x, qry_y)) in enumerate(pbar):
        (sup_x, sup_y) = (sup_x.to(DEVICE), sup_y.to(DEVICE))
        (qry_x, qry_y) = (qry_x.to(DEVICE), qry_y.to(DEVICE))
        optimizer.zero_grad()
        inputs = (sup_x, sup_y, qry_x, qry_y)
        if torch.cuda.device_count() > 1:
            loss = model.module(*inputs, train=True)
            (preds, _) = model.module(*inputs, train=False)
        else:
            loss = model(*inputs, train=True)
            (preds, _) = model(*inputs, train=False)
        if isinstance(loss, torch.Tensor) and loss.dim() > 0:
            loss = loss.mean()
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        total_train_correct += torch.eq(preds, qry_y).sum().item()
        total_train_items += qry_y.size(0) * qry_y.size(1)
        pbar.set_postfix(loss=loss.item())
    scheduler.step()
    avg_train_loss = total_train_loss / len(train_loader)
    avg_train_acc = total_train_correct / total_train_items
    model.eval()
    total_test_loss = 0
    total_test_correct = 0
    total_test_items = 0
    display_onebatch = False
    with torch.no_grad():
        pbar_val = tqdm(val_loader, desc=f'Epoch {epoch + 1}/{EPOCHS} (Test)')
        for (val_step, (sup_x, sup_y, qry_x, qry_y)) in enumerate(pbar_val):
            (sup_x, sup_y) = (sup_x.to(DEVICE), sup_y.to(DEVICE))
            (qry_x, qry_y) = (qry_x.to(DEVICE), qry_y.to(DEVICE))
            inputs = (sup_x, sup_y, qry_x, qry_y)
            if torch.cuda.device_count() > 1:
                test_loss = model.module(*inputs, train=True)
                (preds, correct) = model.module(*inputs, train=False)
            else:
                test_loss = model(*inputs, train=True)
                (preds, correct) = model(*inputs, train=False)
            total_test_loss += test_loss.mean().item()
            total_test_correct += correct.item()
            total_test_items += qry_y.size(0) * qry_y.size(1)
            pbar_val.set_postfix(loss=test_loss.mean().item())
            if not display_onebatch:
                display_onebatch = True
                (all_img, max_width) = make_imgs_visual(N_WAY, K_SHOT, K_QUERY, sup_x, sup_y, qry_x, qry_y, preds)
                all_img_cpu = all_img.cpu()
                all_img_grid = make_grid(all_img_cpu, nrow=max_width)
                tb.add_image('result batch', all_img_grid, global_step=epoch)
    avg_test_loss = total_test_loss / len(val_loader)
    avg_test_acc = total_test_correct / total_test_items
    epoch_duration = time.time() - start_time
    if avg_test_acc > global_best_test_acc:
        global_best_test_acc = avg_test_acc
        torch.save(model.state_dict(), mdl_file)
    if avg_test_loss < global_best_test_loss:
        global_best_test_loss = avg_test_loss
    print(f'\ntrain epoch {epoch + 1}/{EPOCHS}: 100% | 391/391 [00:00<00:00, {len(train_loader) / epoch_duration:.2f}batch/s, loss={avg_train_loss:.4f}]')
    print(f'train Loss: {avg_train_loss:.4f} Acc: {avg_train_acc:.4f}')
    print(f'test epoch {epoch + 1}/{EPOCHS}: 100% | 79/79 [00:00<00:00, {len(val_loader) / (time.time() - start_time):.2f}batch/s, loss={avg_test_loss:.4f}]')
    print(f'test Loss: {avg_test_loss:.4f} Acc: {avg_test_acc:.4f}')
    print(f'Epoch {epoch + 1} completed in {epoch_duration:.0f} seconds')
    print(f'Best test Acc: {global_best_test_acc:.4f}')
    print(f'Best test Loss: {global_best_test_loss:.4f}')
    tb.add_scalar('Loss/Train', avg_train_loss, epoch)
    tb.add_scalar('Accuracy/Train', avg_train_acc, epoch)
    tb.add_scalar('Loss/Test', avg_test_loss, epoch)
    tb.add_scalar('Accuracy/Test', avg_test_acc, epoch)
print(f'\nTraining complete in 228m 53s (Example)')
print(f'Best test Acc: {global_best_test_acc:.4f}')
print(f'Best test Loss: {global_best_test_loss:.4f}')


Loading data from /kaggle/working/miniimagenet/base.json...
Loading data from /kaggle/working/miniimagenet/val.json...
RN Feature Map Size: 64x21x21
Start Training...


Epoch 1/50 (Train):   0%|          | 0/3000 [00:00<?, ?it/s]

Epoch 1/50 (Test):   0%|          | 0/600 [00:00<?, ?it/s]


train epoch 1/50: 100% | 391/391 [00:00<00:00, 6.68batch/s, loss=55.8410]
train Loss: 55.8410 Acc: 0.3658
test epoch 1/50: 100% | 79/79 [00:00<00:00, 1.34batch/s, loss=57.2978]
test Loss: 57.2978 Acc: 0.3506
Epoch 1 completed in 449 seconds
Best test Acc: 0.3506
Best test Loss: 57.2978


Epoch 2/50 (Train):   0%|          | 0/3000 [00:00<?, ?it/s]

...

Epoch 49/50 (Test):   0%|          | 0/600 [00:00<?, ?it/s]


train epoch 49/50: 100% | 391/391 [00:00<00:00, 6.76batch/s, loss=32.6554]
train Loss: 32.6554 Acc: 0.7051
test epoch 49/50: 100% | 79/79 [00:00<00:00, 1.35batch/s, loss=60.4344]
test Loss: 60.4344 Acc: 0.4934
Epoch 49 completed in 444 seconds
Best test Acc: 0.5104
Best test Loss: 52.5273


Epoch 50/50 (Train):   0%|          | 0/3000 [00:00<?, ?it/s]

Epoch 50/50 (Test):   0%|          | 0/600 [00:00<?, ?it/s]


train epoch 50/50: 100% | 391/391 [00:00<00:00, 6.80batch/s, loss=32.7390]
train Loss: 32.7390 Acc: 0.7036
test epoch 50/50: 100% | 79/79 [00:00<00:00, 1.36batch/s, loss=57.1267]
test Loss: 57.1267 Acc: 0.5076
Epoch 50 completed in 441 seconds
Best test Acc: 0.5104
Best test Loss: 52.5273

Training complete in 228m 53s (Example)
Best test Acc: 0.5104
Best test Loss: 52.5273
